In [ ]:
# 1. Instalación de librerías necesarias
!pip install openai-whisper vosk pydub

# 2. Descarga del modelo de lenguaje en español para Vosk (Modelo liviano/eficiente)
!wget -q https://alphacephei.com/vosk/models/vosk-model-small-es-0.42.zip
!unzip -q vosk-model-small-es-0.42.zip -d vosk_model

# 3. Descomprimir archivo de audios
import zipfile
import os

zip_path = 'audios.zip'
extract_path = './audios_mp3'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_path)
    print(f"✅ ¡Éxito! Audios descomprimidos en: {extract_path}")
else:
    print("❌ Error: No se encontró el archivo 'audios.zip' en la raíz de Colab.")

replace vosk_model/vosk-model-small-es-0.42/graph/phones/word_boundary.int? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace vosk_model/vosk-model-small-es-0.42/graph/Gr.fst? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace vosk_model/vosk-model-small-es-0.42/graph/HCLr.fst? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace vosk_model/vosk-model-small-es-0.42/graph/disambig_tid.int? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace vosk_model/vosk-model-small-es-0.42/am/final.mdl? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace vosk_model/vosk-model-small-es-0.42/README? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace vosk_model/vosk-model-small-es-0.42/conf/model.conf? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace vosk_model/vosk-model-small-es-0.42/conf/mfcc.conf? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace vosk_model/vosk-model-small-es-0.42/ivector/final.dubm? [y]es, [n]o, [A]ll, [N]one, [r]ename: y
replace vosk_model/vosk-model-small-es-0.42/ivector/global_cmvn.stats? [y]es, [n]o,

In [ ]:
#Configuración del audio
import glob
from pydub import AudioSegment

wav_path = './audios_wav_16k'
os.makedirs(wav_path, exist_ok=True)

# Tomamos una muestra limitada de hasta 50 audios (.mp3)
mp3_files = sorted(glob.glob(os.path.join(extract_path, '**/*.mp3'), recursive=True))[:50]

print(f"Procesando y estandarizando {len(mp3_files)} archivos de audio...")

for file in mp3_files:
    filename = os.path.basename(file).replace('.mp3', '.wav')
    target_file = os.path.join(wav_path, filename)

    # Conversión técnica: Forzar Mono y canales a 16000Hz (Parámetro de configuración acústica)
    sound = AudioSegment.from_mp3(file)
    sound = sound.set_frame_rate(16000).set_channels(1)
    sound.export(target_file, format="wav")

print(f"✅ Curación completada. 50 audios listos en '{wav_path}'")

Procesando y estandarizando 50 archivos de audio...
✅ Curación completada. 50 audios listos en './audios_wav_16k'


In [ ]:
import time
import json
import whisper
from vosk import Model, KaldiRecognizer
import pandas as pd

# ==========================================
# Cargar Modelos en Memoria
# ==========================================
print("Cargando modelo Whisper (base)...")
model_whisper = whisper.load_model("base")

print("Cargando modelo Vosk (small-es)...")
# Buscamos la carpeta interna que extrajo wget
vosk_folder = glob.glob('./vosk_model/vosk-model-*')[0]
model_vosk = Model(vosk_folder)

# ==========================================
# Contenedor de Resultados (Dataset para Métricas)
# ==========================================
resultados_experimento = []

wav_files = sorted(glob.glob(os.path.join(wav_path, '*.wav')))

print(f"\nIniciando fase experimental sobre {len(wav_files)} audios...")

for idx, file_path in enumerate(wav_files):
    filename = os.path.basename(file_path)

    # ------------------------------------------
    # 1. EJECUCIÓN WHISPER (Con ajuste de parámetros)
    # ------------------------------------------
    start_time = time.time()

    # AJUSTE DE PARÁMETROS DE GENERACIÓN
    # Temperature baja evita alucinaciones; Beam size > 1 explora mejores hipótesis de texto.
    whisper_result = model_whisper.transcribe(
        file_path,
        language='es',
        temperature=0.0,
        beam_size=3,
        patience=1.0
    )

    latency_whisper = time.time() - start_time
    txt_whisper = whisper_result['text'].strip()

    # ------------------------------------------
    # 2. EJECUCIÓN VOSK (Configuración de Arquitectura Lineal)
    # ------------------------------------------
    start_time = time.time()

    wf = open(file_path, "rb")
    # Ignorar las primeras cabeceras del WAV para lectura limpia de bytes
    wf.read(44)

    rec = KaldiRecognizer(model_vosk, 16000)
    rec.SetWords(True) # Habilitar marcas de tiempo por palabra

    txt_vosk_list = []
    while True:
        data = wf.read(4000)
        if len(data) == 0:
            break
        if rec.AcceptWaveform(data):
            res = json.loads(rec.Result())
            if 'text' in res:
                txt_vosk_list.append(res['text'])

    final_res = json.loads(rec.FinalResult())
    if 'text' in final_res:
        txt_vosk_list.append(final_res['text'])

    wf.close()

    latency_vosk = time.time() - start_time
    txt_vosk = " ".join(txt_vosk_list).strip()

    # Guardar traza individual
    resultados_experimento.append({
        'ID_Audio': filename,
        'Transcripción_Whisper': txt_whisper,
        'Latencia_Whisper_Sec': round(latency_whisper, 3),
        'Transcripción_Vosk': txt_vosk,
        'Latencia_Vosk_Sec': round(latency_vosk, 3)
    })

    if (idx + 1) % 10 == 0:
        print(f"Progreso: {idx + 1}/50 audios procesados.")

# Guardar los resultados en un DataFrame de Pandas
df_avance = pd.DataFrame(resultados_experimento)
df_avance.to_csv("avance_transcripciones_50.csv", index=False)
print("\n✅ ¡Fase experimental terminada! Archivo 'avance_transcripciones_50.csv' generado.")

Cargando modelo Whisper (base)...
Cargando modelo Vosk (small-es)...

Iniciando fase experimental sobre 50 audios...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/

Progreso: 10/50 audios procesados.


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/

Progreso: 20/50 audios procesados.


...

Progreso: 30/50 audios procesados.


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/

Progreso: 40/50 audios procesados.


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")
/usr/local/lib/

Progreso: 50/50 audios procesados.

✅ ¡Fase experimental terminada! Archivo 'avance_transcripciones_50.csv' generado.


In [ ]:
#Visualización
# Mostrar las primeras 5 filas del dataset experimental generado
df_avance.head()

,ID_Audio,Transcripción_Whisper,Latencia_Whisper_Sec,Transcripción_Vosk,Latencia_Vosk_Sec
0,spontaneous-speech-es-71834.wav,mi festividad favorita es la Navidad se festej...,11.816,y festividad favorita es la navidad se festeja...,1.081
1,spontaneous-speech-es-71835.wav,¿Qué más cotas son los profuerías en donde vie...,10.605,que mascotas son las preferidas en donde vives...,3.089
2,spontaneous-speech-es-74952.wav,"El pinaca, selva, sana odia, carne rojas.",5.036,espinaca acelga zanahoria carnes rojas,0.687
3,spontaneous-speech-es-74953.wav,en algún momento entre la primaria y la secund...,4.058,en algún momento entre la primaria y secundaria,0.592
4,spontaneous-speech-es-74954.wav,y por mi región normalmente los niños juegan a...,4.179,y por mi región normalmente los niños juegan a...,1.084
